# Analisis sobre el control del brazo robótico

Division de articulaciones :

- Giro de la base 
- Movimiento del hombro 
- Movimiento del codo 
- Movimiento de la muñeca 
- Apertura/cierre de la pinza

### Observación importante

Al tener un brazo de **5 grados de libertad** , el control de cada servo debe ser correcto e intuitivo para el usuario.

En nuestro proyecto tenemos:

- **Entrada:** joystick, potenciómetro u otro dispositivo que el usuario mueve. Si es analógico, necesita ADC.
- **Salida:** señal que la EDU-CIAA genera para controlar cada servo. Señal PWM.

Además, la cantidad de entradas ADC depende del sistema de mando, no directamente de la cantidad de servos.

### Propuesta de control

Una alternativa intuitiva e ideal sería usar:

- Joystick 1 X → Base.
- Joystick 1 Y → Hombro.
- Joystick 2 X → Codo.
- Joystick 2 Y → Muñeca.
- Botón → Abrir/cerrar pinza.

Esta opción permite controlar cuatro movimientos continuos simultáneamente y usar botones para la pinza.

## 1. Problema de las entradas analógicas

En el análisis inicial apareció una posible limitación: la cantidad de entradas ADC que están directamente disponibles en los conectores de la EDU-CIAA.
Según la documentacion, la placa posee 3 pines de entrada analógica(CH1, CH2 y CH3). Esto limita bastante el control del brazo.

### **Alternativa** - Un solo joystick con cambio de modo


- X → un servo.
- Y → otro servo.
- Botón del joystick → cambia el grupo de servos controlados.

Por ejemplo:

```text
Modo 1:
Joystick X → Base
Joystick Y → Hombro

Modo 2:
Joystick X → Codo
Joystick Y → Muñeca
```

La pinza podría controlarse con botones.

### Ventajas

- Menos hardware.
- Menos entradas ADC.
- Implementación sencilla.
- No requiere multiplexor para los cuatro ejes.

### Desventajas

- El operador debe cambiar de modo.
- No permite mover libremente varios ejes a la vez.
- Puede resultar menos cómodo para movimientos precisos.

### Evaluación

Es una buena alternativa para un **primer prototipo o modo de respaldo**, aunque dos joysticks resultarían más naturales para teleoperar cuatro articulaciones.

### **Otra alternativa** — Agregar multiplexor analógico

Se puede incorporar un multiplexor analógico, por ejemplo un **74HC4051**, para seleccionar distintos canales analógicos utilizando una sola entrada ADC.

Conceptualmente:

```text
Joystick 1 X ─┐
Joystick 1 Y ─┤
Joystick 2 X ─┤──> Multiplexor ──> ADC de EDU-CIAA
Joystick 2 Y ─┘
                   ↑
              GPIO de selección
```

La EDU-CIAA seleccionaría un canal mediante líneas digitales, esperaría el tiempo necesario para estabilizar la señal y realizaría la conversión ADC.

### Ventaja

Permite aumentar la cantidad de señales analógicas disponibles sin cambiar la placa principal.

### Desventaja

Agrega hardware y software al proyecto.


# 2. Analisis de pines

Al chequear el documento del pinout de EDU-CIAA-NXP, ademas de estar las entradas analógicas dichas anteriormente, hay ciertos pines donde se pueden configurar para que sea una entrada analógica.



 *Entradas analogicas principales:*

![fig1](<Captura de pantalla (592).png>)

 *Entradas analogicas configurables:*

![alt text](<Captura de pantalla (594).png>)
![alt text](<Captura de pantalla (595).png>)


Esto es posible debido a que en LPC4337 un mismo pin puede tener diferentes funciones, dependiendo de como se lo configure. 

SCU — System Control Unit

Por ejemplo:

                 P4_1
                  │
       ┌──────────┼───────────┐
       │          │           │
      GPIO       ADC        USART

La SCU permite seleccionar cuál de esas funciones tendrá el pin.

Queda pendiente seguir investigando para entender mejor la configuración, programación y el funcionamiento de estos pines.


# 3. Control de los servos

Este es uno de los temas que habrá que estudiar específicamente para el proyecto.
El servo no se controla directamente con el valor del joystick. El programa debe realizar la siguiente conversión:

```text
       ADC del joystick
              ↓
     Filtrado / zona muerta
              ↓
            Mapeo
              ↓
       Ángulo deseado
              ↓
   Señal de control del servo(pwm)
```

## Zona muerta

Los joysticks pueden entregar valores que fluctúan incluso cuando están en reposo.

Por eso conviene establecer una zona alrededor del centro.

Si el valor está dentro de la zona muerta, el sistema puede considerar que el joystick está centrado.

## Mapeo

Conceptualmente:

```text
ADC mínimo  → ángulo mínimo
ADC centro  → posición central
ADC máximo  → ángulo máximo
```

Los límites reales deberán determinarse experimentalmente para cada articulación.